# Ch.4 — Polynomial Features & Feature Engineering

**Goal**: Capture non-linear patterns to improve MAE from $55k → $48k.

| What | Value |
|------|-------|
| Dataset | California Housing (20,640 districts, 8 features) |
| Ch.2 baseline | ~$55k MAE (8 linear features) |
| This chapter target | ~$48k MAE (polynomial features) |
| Grand Challenge target | <$40k MAE |

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

In [ ]:
# ── Load and split ─────────────────────────────────────────────────────────
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = housing.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples")

## Why Linear Features Aren't Enough

Let's visualize the non-linear relationship that a straight line can't capture.

In [ ]:
# ── Non-linearity visualization ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# MedInc vs Value — curved relationship
axes[0].scatter(X_train['MedInc'], y_train, alpha=0.1, s=5, c='steelblue')
# Add linear fit line
m, b = np.polyfit(X_train['MedInc'], y_train, 1)
x_line = np.linspace(0, 15, 100)
axes[0].plot(x_line, m * x_line + b, 'r--', linewidth=2, label='Linear fit')
# Add polynomial fit
coeffs = np.polyfit(X_train['MedInc'], y_train, 2)
axes[0].plot(x_line, np.polyval(coeffs, x_line), 'g-', linewidth=2, label='Quadratic fit')
axes[0].set_xlabel('MedInc ($10k)', fontsize=12)
axes[0].set_ylabel('MedHouseVal ($100k)', fontsize=12)
axes[0].set_title('MedInc vs Value\n(Notice the curve!)', fontsize=12)
axes[0].legend(fontsize=10)

# Latitude vs Value — U-shaped
axes[1].scatter(X_train['Latitude'], y_train, alpha=0.1, s=5, c='steelblue')
m2, b2 = np.polyfit(X_train['Latitude'], y_train, 1)
x_lat = np.linspace(32, 42, 100)
axes[1].plot(x_lat, m2 * x_lat + b2, 'r--', linewidth=2, label='Linear')
coeffs2 = np.polyfit(X_train['Latitude'], y_train, 2)
axes[1].plot(x_lat, np.polyval(coeffs2, x_lat), 'g-', linewidth=2, label='Quadratic')
axes[1].set_xlabel('Latitude', fontsize=12)
axes[1].set_title('Latitude vs Value\n(Coastal effect)', fontsize=12)
axes[1].legend(fontsize=10)

# Residuals from linear model — should show pattern
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_train)
lr = LinearRegression().fit(X_tr_s, y_train)
resid = y_train - lr.predict(X_tr_s)
axes[2].scatter(lr.predict(X_tr_s), resid, alpha=0.1, s=5, c='steelblue')
axes[2].axhline(y=0, color='red', linewidth=1.5, linestyle='--')
axes[2].set_xlabel('Predicted Value', fontsize=12)
axes[2].set_ylabel('Residual', fontsize=12)
axes[2].set_title('Linear Model Residuals\n(Pattern = non-linearity!)', fontsize=12)

plt.tight_layout()
plt.savefig('img/nonlinearity_evidence.png', dpi=150, bbox_inches='tight')
plt.show()

## Polynomial Feature Expansion

Transform 8 features → 44 (degree 2) by adding squared and interaction terms.

In [ ]:
# ── Feature expansion demo ────────────────────────────────────────────────
poly = PolynomialFeatures(degree=2, include_bias=False)
X_demo = poly.fit_transform(X_train.values[:1])  # Single sample

feature_names = poly.get_feature_names_out(housing.feature_names)

print(f"Raw features: {len(housing.feature_names)}")
print(f"Polynomial features: {len(feature_names)}")
print(f"\nBreakdown:")
print(f"  Original:     {sum(1 for f in feature_names if '^' not in f and ' ' not in f)}")
print(f"  Squared (x²): {sum(1 for f in feature_names if '^2' in f)}")
print(f"  Cross (xᵢxⱼ): {sum(1 for f in feature_names if ' ' in f and '^' not in f)}")
print(f"\nSample polynomial features:")
for name in feature_names[:8]:
    print(f"  {name}")
print("  ...")
for name in feature_names[-5:]:
    print(f"  {name}")

## Numerical Walkthrough: Raw Values → Polynomial Features

The expansion is concrete: for a single district, every raw number spawns squared and cross-product columns.  
Below we trace **exactly one district** through the transformation and show how each weight multiplies its paired feature.

In [ ]:
# ── Numerical walkthrough ─────────────────────────────────────────────────
# Step 1: pick one real district and show the mapping in plain numbers

sample = X_train.iloc[0]
actual_price = y_train.iloc[0] * 100_000

print("=" * 62)
print("  DISTRICT SNAPSHOT")
print("=" * 62)
print(f"  MedInc    = {sample['MedInc']:.4f}   (median income, ×$10k)")
print(f"  HouseAge  = {sample['HouseAge']:.4f}")
print(f"  AveRooms  = {sample['AveRooms']:.4f}")
print(f"  Actual price: ${actual_price:,.0f}")

# ── Step 2: 1-feature polynomial mapping (MedInc only) ───────────────────
x = sample['MedInc']
print("\n── 1-Feature Polynomial Expansion (degree=2) ─────────────────")
print(f"  Raw value:    x          = {x:.4f}")
print(f"  x₁  = x      = MedInc   = {x:.4f}   ← identical to raw")
print(f"  x₂  = x²     = MedInc²  = {x**2:.4f}   ← squared")
print(f"\n  Prediction form: ŷ = b + w₁·x₁ + w₂·x₂")
print(f"                      = b + w₁·{x:.2f} + w₂·{x**2:.2f}")

# ── Step 3: 3-feature expansion with real numbers ─────────────────────────
selected = ['MedInc', 'HouseAge', 'AveRooms']
vals = sample[selected].values.reshape(1, -1)

poly3 = PolynomialFeatures(degree=2, include_bias=False)
expanded = poly3.fit_transform(vals)
names3   = poly3.get_feature_names_out(selected)

print("\n── 3-Feature Polynomial Expansion (degree=2) ─────────────────")
print(f"  {'Feature name':28s}  {'Raw calc':24s}  {'Value':>10}")
print(f"  {'-'*28}  {'-'*24}  {'-'*10}")

for name, val in zip(names3, expanded[0]):
    if name in selected:
        raw_calc = f"= {sample[name]:.4f}"
        category = "original"
    elif '^2' in name:
        base = name.replace('^2', '')
        raw_calc = f"= {sample[base]:.4f}² "
        category = "squared"
    else:
        parts = name.split(' ')
        raw_calc = f"= {sample[parts[0]]:.3f} × {sample[parts[1]]:.3f}"
        category = "interaction"
    print(f"  {name:28s}  {raw_calc:24s}  {val:>10.4f}   [{category}]")

# ── Step 4: show how weight × feature → contribution ─────────────────────
print("\n── Contribution of each feature to ŷ (using fitted model) ────")
# Fit a small poly model on MedInc, HouseAge, AveRooms for clarity
X_small = X_train[selected]
pipe_small = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('lr', LinearRegression())
])
pipe_small.fit(X_small, y_train)

w = pipe_small.named_steps['lr'].coef_
b = pipe_small.named_steps['lr'].intercept_

# transform single sample through scaler
x_scaled_single = pipe_small.named_steps['scaler'].transform(expanded)
contributions = w * x_scaled_single[0]

print(f"  {'Feature':28s}  {'Std value':>10}  {'Weight':>8}  {'Contribution':>13}")
print(f"  {'-'*28}  {'-'*10}  {'-'*8}  {'-'*13}")
for name, sv, wi, ci in zip(names3, x_scaled_single[0], w, contributions):
    print(f"  {name:28s}  {sv:>10.4f}  {wi:>8.4f}  {ci:>13.4f}")
print(f"  {'Bias (intercept)':28s}  {'':>10}  {'':>8}  {b:>13.4f}")
y_hat = contributions.sum() + b
print(f"  {'':28s}  {'':>10}  {'':>8}  {'─'*13}")
print(f"  {'ŷ (sum)':28s}  {'':>10}  {'':>8}  {y_hat:>13.4f}  (×$100k)")
print(f"  Predicted: ${y_hat*100_000:,.0f}   |   Actual: ${actual_price:,.0f}")

## Animated Weight Evolution: How w₁ and w₂ Learn

Using MedInc only (1 raw feature → 2 polynomial features: x and x²), we run gradient descent from scratch and record every weight update.  
Three panels track:  
- **Left**: the fitted curve draped over the data — starts flat, bends into shape  
- **Middle**: the weight trajectory in (w₁, w₂) space — shows where gradient descent walks  
- **Right**: the MSE loss dropping toward its minimum  

The weight table printed below each frame shows the exact values of b, w₁(x), and w₂(x²) at that epoch.

In [ ]:
import matplotlib.animation as animation
from matplotlib.animation import PillowWriter
import os

# ── 1. Build 1-D polynomial dataset (MedInc → Value) ─────────────────────
np.random.seed(42)
sample_idx = np.random.choice(len(X_train), 3000, replace=False)
x_raw = X_train['MedInc'].values[sample_idx]          # shape (3000,)
y_raw = y_train.values[sample_idx]                    # shape (3000,)

# Polynomial feature matrix: columns = [x, x²]
X_poly = np.column_stack([x_raw, x_raw ** 2])         # (n, 2)

# Standardize both columns (fit on this subset)
mu_p  = X_poly.mean(axis=0)
std_p = X_poly.std(axis=0)
X_s   = (X_poly - mu_p) / std_p                       # (n, 2)

# Augment with bias column
n = len(y_raw)
X_b = np.column_stack([np.ones(n), X_s])              # (n, 3): [1, x_std, x²_std]

# ── 2. Gradient descent — record weight history ───────────────────────────
w       = np.zeros(3)                                 # [bias, w1, w2]
lr_rate = 0.05
epochs  = 250
history = [w.copy()]
losses  = []

for _ in range(epochs):
    pred      = X_b @ w
    residuals = pred - y_raw
    losses.append((residuals ** 2).mean())
    grad = (2 / n) * (X_b.T @ residuals)
    w    = w - lr_rate * grad
    history.append(w.copy())

history = np.array(history)   # (251, 3)

# ── 3. Build the animation ────────────────────────────────────────────────
x_plot     = np.linspace(0, 15, 300)
X_plot_raw = np.column_stack([x_plot, x_plot ** 2])
X_plot_s   = (X_plot_raw - mu_p) / std_p
X_plot_b   = np.column_stack([np.ones(len(x_plot)), X_plot_s])

fig = plt.figure(figsize=(15, 5))
fig.suptitle("Polynomial Regression — Weight Evolution (MedInc + MedInc²)",
             fontsize=13, fontweight='bold', y=1.02)
gs  = fig.add_gridspec(1, 3, wspace=0.35)
ax1 = fig.add_subplot(gs[0])   # fitted curve
ax2 = fig.add_subplot(gs[1])   # weight trajectory
ax3 = fig.add_subplot(gs[2])   # loss curve

# Panel 1 — scatter + curve
ax1.scatter(x_raw, y_raw, alpha=0.07, s=6, c='steelblue', label='Data')
curve_line, = ax1.plot([], [], 'r-', linewidth=2.0, label='ŷ = b+w₁x+w₂x²')
epoch_lbl   = ax1.text(0.04, 0.96, '', transform=ax1.transAxes,
                        fontsize=9, va='top', family='monospace',
                        bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8))
ax1.set_xlim(0, 15);  ax1.set_ylim(-0.5, 5.5)
ax1.set_xlabel('MedInc ($10k)');  ax1.set_ylabel('MedHouseVal ($100k)')
ax1.set_title('Fitted Curve'); ax1.legend(fontsize=8)

# Panel 2 — weight space
w_final = history[-1]
ax2.scatter([w_final[1]], [w_final[2]], marker='*', s=200, color='gold',
            zorder=8, label='Final w', edgecolors='black', linewidths=0.5)
ax2.axhline(0, color='gray', lw=0.5); ax2.axvline(0, color='gray', lw=0.5)
w1_lo, w1_hi = history[:, 1].min() - 0.1, history[:, 1].max() + 0.1
w2_lo, w2_hi = history[:, 2].min() - 0.1, history[:, 2].max() + 0.1
ax2.set_xlim(w1_lo, w1_hi);  ax2.set_ylim(w2_lo, w2_hi)
ax2.set_xlabel('w₁  (weight of x / MedInc)');  ax2.set_ylabel('w₂  (weight of x² / MedInc²)')
ax2.set_title('Weight Space Trajectory')
path_line, = ax2.plot([], [], '-', color='tomato', alpha=0.5, lw=1.2)
path_dot,  = ax2.plot([], [], 'o', color='red', markersize=7, zorder=7)
ax2.legend(fontsize=8)

# Panel 3 — loss
ax3.plot(range(len(losses)), losses, color='cornflowerblue', lw=0.9, alpha=0.5)
loss_dot, = ax3.plot([], [], 'ro', markersize=6, zorder=5)
ax3.set_xlim(0, epochs); ax3.set_ylim(0, losses[0] * 1.05)
ax3.set_xlabel('Epoch');  ax3.set_ylabel('MSE Loss')
ax3.set_title('Loss Curve')

# Weight table as figure text (updated each frame)
wtab = fig.text(0.01, -0.05, '', fontsize=9, family='monospace',
                va='top', ha='left',
                bbox=dict(boxstyle='round,pad=0.4', fc='#f0f0f0', alpha=0.9))

def _init():
    curve_line.set_data([], [])
    path_line.set_data([], [])
    path_dot.set_data([], [])
    loss_dot.set_data([], [])
    epoch_lbl.set_text('')
    wtab.set_text('')
    return curve_line, path_line, path_dot, loss_dot, epoch_lbl, wtab

step   = 2
frames = list(range(0, epochs + 1, step))

def _update(frame):
    wc = history[frame]

    # Curve
    y_hat = X_plot_b @ wc
    curve_line.set_data(x_plot, y_hat)

    # Weight path
    path_line.set_data(history[:frame + 1, 1], history[:frame + 1, 2])
    path_dot.set_data([wc[1]], [wc[2]])

    # Loss dot
    li = min(frame, len(losses) - 1)
    loss_dot.set_data([frame], [losses[li]])

    epoch_lbl.set_text(f'Epoch {frame}/{epochs}\nLoss={losses[li]:.4f}')

    # Weight table
    wtab.set_text(
        f"Epoch {frame:3d}  │  "
        f"b = {wc[0]:+.4f}  │  "
        f"w₁(x) = {wc[1]:+.4f}  │  "
        f"w₂(x²) = {wc[2]:+.4f}"
    )
    return curve_line, path_line, path_dot, loss_dot, epoch_lbl, wtab

anim = animation.FuncAnimation(fig, _update, frames=frames,
                                init_func=_init, blit=True, interval=80)

plt.tight_layout()

os.makedirs('img', exist_ok=True)
anim.save('img/polynomial_weight_evolution.gif', writer=PillowWriter(fps=14))
plt.close()
print("Animation saved → img/polynomial_weight_evolution.gif")

# Print final weight summary
wf = history[-1]
print(f"\nFinal weights after {epochs} epochs:")
print(f"  b     = {wf[0]:+.4f}")
print(f"  w₁(x)  = {wf[1]:+.4f}   ← MedInc (standardized)")
print(f"  w₂(x²) = {wf[2]:+.4f}   ← MedInc² (standardized)")
print(f"\nPrediction for a district with MedInc=3.5:")
x_eg     = 3.5
x_eg_raw = np.array([[x_eg, x_eg ** 2]])
x_eg_s   = (x_eg_raw - mu_p) / std_p
x_eg_b   = np.concatenate([[1], x_eg_s[0]])
y_eg     = x_eg_b @ wf
print(f"  x₁ = MedInc    = {x_eg:.2f}")
print(f"  x₂ = MedInc²   = {x_eg**2:.2f}")
print(f"  ŷ  = {wf[0]:+.4f} + {wf[1]:+.4f}×{x_eg_s[0,0]:.4f} + {wf[2]:+.4f}×{x_eg_s[0,1]:.4f}")
print(f"     = {y_eg:.4f}  →  ${y_eg*100_000:,.0f}")

## Degree Sweep — Finding the Sweet Spot

Sweep polynomial degree from 1 to 4 and track train/test MAE.
The **bias-variance trade-off** in action: higher degree → lower train error but eventually higher test error.

In [ ]:
# ── Degree sweep ──────────────────────────────────────────────────────────
degrees = [1, 2, 3]
results = []

for deg in degrees:
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=deg, include_bias=False)),
        ('scaler', StandardScaler()),
        ('model', LinearRegression())
    ])
    pipe.fit(X_train, y_train)
    
    train_mae = mean_absolute_error(y_train, pipe.predict(X_train)) * 100_000
    test_mae = mean_absolute_error(y_test, pipe.predict(X_test)) * 100_000
    n_feats = pipe.named_steps['poly'].n_output_features_
    r2 = r2_score(y_test, pipe.predict(X_test))
    
    results.append({'degree': deg, 'n_features': n_feats,
                    'train_mae': train_mae, 'test_mae': test_mae, 'r2': r2})
    
    gap = test_mae - train_mae
    flag = " ⚠️ OVERFITTING" if gap > 5000 else ""
    print(f"Degree {deg}: {n_feats:4d} features | "
          f"Train ${train_mae:,.0f} | Test ${test_mae:,.0f} | "
          f"Gap ${gap:,.0f} | R²={r2:.4f}{flag}")

results_df = pd.DataFrame(results)

In [ ]:
# ── Bias-variance plot ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(results_df['degree'], results_df['train_mae'], 'b-o', linewidth=2,
        markersize=8, label='Train MAE', zorder=5)
ax.plot(results_df['degree'], results_df['test_mae'], 'r-s', linewidth=2,
        markersize=8, label='Test MAE', zorder=5)
ax.axhline(y=40_000, color='green', linestyle='--', linewidth=1.5,
           label='🎯 $40k Target')

# Annotate
for _, row in results_df.iterrows():
    ax.annotate(f"${row['test_mae']:,.0f}\n({int(row['n_features'])} feats)",
                xy=(row['degree'], row['test_mae']),
                textcoords='offset points', xytext=(15, 10), fontsize=9)

ax.set_xlabel('Polynomial Degree', fontsize=12)
ax.set_ylabel('MAE ($)', fontsize=12)
ax.set_title('Bias-Variance Trade-off: Polynomial Degree Sweep\n'
             'Sweet spot at degree=2 (captures curves without overfitting)', fontsize=14)
ax.legend(fontsize=11)
ax.set_xticks(degrees)
plt.tight_layout()
plt.savefig('img/degree_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

## Top Polynomial Features

Which polynomial features contribute most? After standardization, largest absolute weights = most important.

In [ ]:
# ── Top polynomial features ───────────────────────────────────────────────
pipe_d2 = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])
pipe_d2.fit(X_train, y_train)

poly_names = pipe_d2.named_steps['poly'].get_feature_names_out(housing.feature_names)
weights = pipe_d2.named_steps['model'].coef_

feat_imp = pd.DataFrame({'feature': poly_names, 'weight': weights})
feat_imp['abs_weight'] = feat_imp['weight'].abs()
top15 = feat_imp.nlargest(15, 'abs_weight')

fig, ax = plt.subplots(figsize=(12, 7))
colors = ['#d4edda' if w > 0 else '#f8d7da' for w in top15['weight']]
bars = ax.barh(range(len(top15)), top15['weight'].values, color=colors, edgecolor='gray')
ax.set_yticks(range(len(top15)))
ax.set_yticklabels(top15['feature'].values, fontsize=10)
ax.set_xlabel('Standardized Weight', fontsize=12)
ax.set_title('Top 15 Polynomial Features by Weight Magnitude\n'
             'Notice interaction terms (x₁ x₂) capturing cross-effects', fontsize=14)
ax.axvline(x=0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig('img/top_polynomial_features.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTop 5 polynomial features:")
for _, row in top15.head().iterrows():
    print(f"  {row['feature']:30s}: {row['weight']:+.4f}")

## Interaction Effects Visualization

Show how MedInc × Latitude interaction captures the coastal premium.

In [ ]:
# ── Interaction effect: MedInc × Latitude ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Color scatter by value
scatter = axes[0].scatter(X_train['MedInc'], X_train['Latitude'],
                          c=y_train, cmap='RdYlGn', alpha=0.4, s=10,
                          vmin=0, vmax=5)
axes[0].set_xlabel('MedInc ($10k)', fontsize=12)
axes[0].set_ylabel('Latitude', fontsize=12)
axes[0].set_title('MedInc × Latitude colored by House Value\n'
                  '(Top-right = rich + coastal = most expensive)', fontsize=12)
plt.colorbar(scatter, ax=axes[0], label='MedHouseVal ($100k)')

# Show interaction feature
interaction = X_train['MedInc'] * X_train['Latitude']
axes[1].scatter(interaction, y_train, alpha=0.1, s=5, c='steelblue')
axes[1].set_xlabel('MedInc × Latitude (interaction term)', fontsize=12)
axes[1].set_ylabel('MedHouseVal ($100k)', fontsize=12)
axes[1].set_title('Interaction Feature vs Value\n'
                  '(Captures the coastal income premium)', fontsize=12)

plt.tight_layout()
plt.savefig('img/interaction_effect.png', dpi=150, bbox_inches='tight')
plt.show()

## Cross-Validation Stability Check

5-fold CV ensures our $48k MAE isn't a lucky split.

In [ ]:
# ── Cross-validation ──────────────────────────────────────────────────────
pipe_cv = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

cv_scores = cross_val_score(pipe_cv, X_train, y_train,
                            cv=5, scoring='neg_mean_absolute_error')
cv_maes = -cv_scores * 100_000

print("5-Fold Cross-Validation (degree=2):")
for i, mae in enumerate(cv_maes, 1):
    print(f"  Fold {i}: ${mae:,.0f}")
print(f"  Mean:   ${cv_maes.mean():,.0f} (±${cv_maes.std():,.0f})")
print(f"  Target: <$40,000")
print(f"  Gap:    ${cv_maes.mean() - 40_000:,.0f}")

## Progress Summary

In [ ]:
# ── Progress comparison ───────────────────────────────────────────────────
# Recompute baselines
sc1 = StandardScaler()
lr1 = LinearRegression().fit(X_train[['MedInc']], y_train)
mae_ch1 = mean_absolute_error(y_test, lr1.predict(X_test[['MedInc']])) * 100_000

sc2 = StandardScaler()
X_tr_s2 = sc2.fit_transform(X_train)
lr2 = LinearRegression().fit(X_tr_s2, y_train)
mae_ch2 = mean_absolute_error(y_test, lr2.predict(sc2.transform(X_test))) * 100_000

mae_ch3 = mean_absolute_error(y_test, pipe_d2.predict(X_test)) * 100_000

print("═" * 60)
print("  PROGRESS CHECK: Ch.1 → Ch.2 → Ch.3")
print("═" * 60)
print(f"  {'Chapter':<25} {'MAE':>12} {'Features':>10} {'R²':>8}")
print(f"  {'-'*25} {'-'*12} {'-'*10} {'-'*8}")
print(f"  {'Ch.1 (1 feature)':<25} {'${:,.0f}'.format(mae_ch1):>12} {'1':>10} {r2_score(y_test, lr1.predict(X_test[['MedInc']])):>8.4f}")
print(f"  {'Ch.2 (8 features)':<25} {'${:,.0f}'.format(mae_ch2):>12} {'8':>10} {r2_score(y_test, lr2.predict(sc2.transform(X_test))):>8.4f}")
print(f"  {'Ch.3 (polynomial d=2)':<25} {'${:,.0f}'.format(mae_ch3):>12} {'44':>10} {r2_score(y_test, pipe_d2.predict(X_test)):>8.4f}")
print(f"  {'-'*25} {'-'*12} {'-'*10} {'-'*8}")
print(f"  {'🎯 Target':<25} {'<$40,000':>12}")
print("═" * 60)
print("\n→ Next: Ch.4 adds regularization to push past the $40k target")

## Exercises

1. **Interaction-only**: Use `PolynomialFeatures(interaction_only=True)` to keep only cross terms (no $x^2$). How does MAE compare?
2. **Single polynomial feature**: Add ONLY `MedInc²` to the 8 raw features (manual column). How much does this single feature help?
3. **Degree 3 overfitting**: Fit degree 3 and plot train/test MAE over increasing training set sizes (learning curve). Where does overfitting start?

In [ ]:
# ── Exercise 1 scaffold: interaction_only=True ────────────────────────────
# TODO: Build pipeline with interaction_only=True, compare MAE
pass

In [ ]:
# ── Exercise 2 scaffold: manual MedInc² feature ───────────────────────────
# TODO: Add MedInc² column manually, fit, compare to degree=2 pipeline
pass

In [ ]:
# ── Exercise 3 scaffold: learning curve for degree 3 ──────────────────────
# TODO: Fit degree 3 with increasing training sizes, plot learning curve
pass